# การสร้าง Reply Prompt ตาม Paper EMNLP 2025 (1664)
อิงจากเปเปอร์ `2025.emnlp-main.1664.pdf` ในหัวข้อ **4.1 Prompt Design** โดยนำค่า Dissonance จาก `compare_dissonance.ipynb` มาใช้ใน prompt 3 วิธี:
1. **Base Prompting**: ใส่แค่บทสนทนา (text dialogue) ตามปกติ
2. **Emotion-Enhanced Prompting**: เสริมข้อมูลเรื่อง emotion (valence, arousal) เติมเข้าไปใน prompt
3. **Dissonance-Based Prompting**: ถ้ามี Dissonance ระหว่าง text กับ speech จะระบุเงื่อนไข "emotional dissonance" ในการให้โมเดลตอบกลับ


In [1]:
import pandas as pd
import numpy as np

# โหลดข้อมูล VAD Speech
speech_csv = r"c:\Users\Legion 5 Pro\OneDrive\Documents\Graduate research\test\wagner\dialogue_2\emotion_results_wavlm_all.csv"
df_speech = pd.read_csv(speech_csv)

# โหลดข้อมูล VAD Text 
text_csv = r"c:\Users\Legion 5 Pro\OneDrive\Documents\Graduate research\test\own_script\dialogue_2\dialogue_2_vad_text.csv"
df_text = pd.read_csv(text_csv)

# ทำการ Scale และคำนวณ Dissonance แบบเดียวกับ compare_dissonance.ipynb
speech = df_speech[df_speech["dialogue_id"] == 2].sort_values("utterance_id").reset_index(drop=True)
text   = df_text.sort_values("utterance_id").reset_index(drop=True)

# Scale ไปช่วง [-1, 1]
speech_aro_s = 2 * np.clip(speech["arousal"].values, 0.0, 1.0) - 1
speech_val_s = 2 * np.clip(speech["valence"].values, 0.0, 1.0) - 1

text_aro_s = 2 * ((np.clip(text["arousal_text"].values, 1.0, 5.0) - 1.0) / 4.0) - 1
text_val_s = 2 * ((np.clip(text["valence_text"].values, 1.0, 5.0) - 1.0) / 4.0) - 1

# คำนวณ Dissonance
delta_arousal = np.abs(speech_aro_s - text_aro_s)
delta_valence = np.abs(speech_val_s - text_val_s)

# Threshold = 0.5 ตาม Paper section 3.1
ARO_THR = 0.5
VAL_THR = 0.5
dissonant_any = (delta_arousal > ARO_THR) | (delta_valence > VAL_THR)

# นำข้อมูลที่จำเป็นมารวมกัน
df = pd.DataFrame({
    "utterance_id": text["utterance_id"],
    "text": text["text"],
    "speech_arousal": speech_aro_s,
    "speech_valence": speech_val_s,
    "text_arousal": text_aro_s,
    "text_valence": text_val_s,
    "delta_arousal": delta_arousal,
    "delta_valence": delta_valence,
    "dissonance": dissonant_any.astype(bool),
})


df.head()


FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\Legion 5 Pro\\OneDrive\\Documents\\Graduate research\\test\\wagner\\dialogue_2\\emotion_results_wavlm_all.csv'

## Prompt Design Methods
เราจะสร้าง 3 functions แทนแต่ละวิธีใน Paper section 4.1

In [2]:
FORMAT_INSTRUCTION = """Your response must be in the form:
TechniqueName: [Your Response]

Use a brief, conventional CBT or psychotherapy technique label as TechniqueName
(e.g., Cognitive Restructuring, Reframing Technique, Socratic Questioning,
Emotional Awareness and Validation, etc.)."""


# Base Prompting
def base_prompting(context: str) -> str:
    prompt = f"""You are a mental health professional interacting with a patient.
Based on the dialogue below, generate a brief therapeutic intervention that is
clinically appropriate and context-aware.

Dialogue:
{context}

{FORMAT_INSTRUCTION}
"""
    return prompt


# Emotion-Enhanced Prompting (continuous valence–arousal from speech)
def emotion_enhanced_prompting(context: str, arousal: float, valence: float) -> str:
    prompt = f"""You are a mental health professional interacting with a patient.
Based on the dialogue below, generate a brief therapeutic intervention that is
clinically appropriate and sensitive to the patient's emotional state.

For the current patient utterance, continuous emotion estimates from speech
are provided as valence–arousal scores.

Dialogue:
{context}
Patient (speech emotion - arousal: {arousal:.2f}, valence: {valence:.2f}): ...

{FORMAT_INSTRUCTION}
"""
    return prompt


# Dissonance-Based Prompting
def dissonance_based_prompting(context: str, has_dissonance: bool) -> str:
    if has_dissonance:
        dissonance_tag = "emotional dissonance – Patient:"
        extra_instruction = (
            "In this case, you detect emotional dissonance: the patient's tone "
            "sounds different from what their words explicitly express. "
            "Pay special attention to this mismatch when formulating your intervention."
        )
    else:
        dissonance_tag = "Patient:"
        extra_instruction = (
            "In this case, you do not detect emotional dissonance. Base your "
            "intervention primarily on the explicit content of the dialogue."
        )

    prompt = f"""You are a mental health professional interacting with a patient.
Based on the dialogue below, generate a brief therapeutic intervention that is
clinically appropriate and attentive to how the patient both speaks and feels.

{extra_instruction}

Dialogue:
{context}
{dissonance_tag} ...

{FORMAT_INSTRUCTION}
"""
    return prompt


## ตัวอย่างการนำไปใช้ (Generating Replies)
ในบล็อกนี้เราจะจำลองการรับ Utterance ล่าสุดของ Patient แล้วใช้ 3 วิธีเพื่อสร้าง prompt
หลังจากได้ Prompt สามารถนำเอา Prompt ไปใช้กับโมเดลอย่างเช่น OpenAI GPT-4, Llama 3 หรืออื่นๆ ได้

In [3]:
# ดึงประโยคแรกมาเทส
row = df.iloc[0]

dialogue_context = f"Patient: {row['text']}"

print("=== 1. Base Prompting ===")
print(base_prompting(dialogue_context))
print("-" * 50)

print("=== 2. Emotion-Enhanced Prompting ===")
print(emotion_enhanced_prompting(dialogue_context, row['speech_arousal'], row['speech_valence']))
print("-" * 50)

print("=== 3. Dissonance-Based Prompting ===")
print(dissonance_based_prompting(dialogue_context, row['dissonance']))
print("-" * 50)


=== 1. Base Prompting ===
You are a mental health professional interacting with a patient.
Based on the dialogue below, generate a brief therapeutic intervention that is
clinically appropriate and context-aware.

Dialogue:
Patient: Why are you bothering me? What's the problem?

Your response must be in the form:
TechniqueName: [Your Response]

Use a brief, conventional CBT or psychotherapy technique label as TechniqueName
(e.g., Cognitive Restructuring, Reframing Technique, Socratic Questioning,
Emotional Awareness and Validation, etc.).

--------------------------------------------------
=== 2. Emotion-Enhanced Prompting ===
You are a mental health professional interacting with a patient.
Based on the dialogue below, generate a brief therapeutic intervention that is
clinically appropriate and sensitive to the patient's emotional state.

For the current patient utterance, continuous emotion estimates from speech
are provided as valence–arousal scores.

Dialogue:
Patient: Why are you bo

## The Generator Section (Hugging Face API / Local Models)
ในบล็อกนี้เราจะนำข้อมูลข้อความและ Prompts ที่เราสร้างจากฟังก์ชันด้านบน มาป้อนให้กับ LLMs รุ่นอย่าง `meta-llama/Meta-Llama-3-8B-Instruct` หรือที่ใกล้เคียงกันครับ
ด้วยข้อจำกัด GPU อย่าง RTX 4060 (8GB VRAM) เราจะทำงานร่วมกับ Quantization 4-Bit ของ BitsAndBytesConfig ผ่านไลบรารี Transformers 

คุณจะต้องติดตั้ง:
`pip install torch transformers accelerate bitsandbytes`


In [4]:
import bitsandbytes as bnb
print(bnb.__version__)

0.49.2


### Load Meta-Llama-3-8B-Instruct Model from huggingface

In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# เลือกใช้ Quantized Model ตามที่สเปคเครื่องพอรับไหวครับ
model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

# แบบมี quantile

# ตั้งค่า 4-Bit เพื่อลดอัตรากิน VRAM เหลือประมาณ ~5-6 GB 
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

print(f"Loading {model_id} in 4-bit...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=quantization_config,
)
print("Model Loaded Successfully!")

Loading meta-llama/Meta-Llama-3-8B-Instruct in 4-bit...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Model Loaded Successfully!


### Generate reply dialogue function

In [6]:
def generate_reply(prompt_text: str) -> str:
    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful and experienced mental health professional. "
                "Based on the patient dialogue, provide brief, empathetic, and "
                "specific therapeutic interventions as instructed."
            ),
        },
        {"role": "user", "content": prompt_text},
    ]

    # ให้ได้ BatchEncoding แล้วดึง input_ids ออกมา
    enc = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
    )
    # enc เป็น dict-like: {"input_ids": ..., "attention_mask": ...}
    enc = {k: v.to(model.device) for k, v in enc.items()}
    input_ids = enc["input_ids"]

    terminators = [
        tokenizer.eos_token_id,
        tokenizer.convert_tokens_to_ids("<|eot_id|>"),
    ]

    output_ids = model.generate(
        **enc,                      # ให้ model เห็นทั้ง input_ids และ attention_mask
        max_new_tokens=150,
        eos_token_id=terminators,   # ถ้า error ก็เปลี่ยนเป็น int เดียว
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        temperature=0.6,
        top_p=0.9,
    )

    generated_tokens = output_ids[0, input_ids.shape[-1]:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True)


### Generate dialogue outputs

In [7]:
print("============= OUTPUT =============")

print("\n[Context - Patient Utterance]")
print(dialogue_context)

print("\n[1. Base Prompting Reply]")
print(generate_reply(base_prompting(dialogue_context)))

print("\n[2. Emotion-Enhanced Prompting Reply]")
print(generate_reply(
    emotion_enhanced_prompting(
        dialogue_context,
        row["speech_arousal"],
        row["speech_valence"],
    )
))

print("\n[3. Dissonance-Based Prompting Reply]")
print(generate_reply(
    dissonance_based_prompting(
        dialogue_context,
        row["dissonance"],
    )
))


============= OUTPUT =============

[Context - Patient Utterance]
Patient: Why are you bothering me? What's the problem?

[1. Base Prompting Reply]
Technique: Emotional Awareness and Validation

"Ah, I understand that you're feeling frustrated and possibly overwhelmed by our conversation. It sounds like you're feeling bothered by my presence, and that's a natural reaction. Can you help me understand what's making you feel this way? Is it something specific I said or did? Sometimes, our emotions can be triggered by a particular situation or experience. Let's take a moment to acknowledge and validate your feelings. It's okay to feel upset or frustrated, and I'm here to support you."

[2. Emotion-Enhanced Prompting Reply]
TechniqueName: Emotional Awareness and Validation

"Ah, I sense that you're feeling quite frustrated and upset right now. It sounds like you're feeling bothered and wondering what's the point of our conversation. I want to acknowledge that it can be really uncomfortable 

# ทำ reply.ipynb ให้เจน 3 แบบ: Base / Emotion-Enhanced / Dissonance-Based

## โหลดข้อมูล + เตรียมตาราง

In [3]:
import os
import json
import pandas as pd
from openai import OpenAI

BASE_DIR = "/home/patsakornt/work/test"
OWN_SCRIPT_DIR = os.path.join(BASE_DIR, "dissonance", "own_script", "dialogue_2")

# 1) โหลด dissonance (จาก df_dissonance ที่เซฟไว้)
diss_path = os.path.join(OWN_SCRIPT_DIR, "dialogue_2_dissonance.csv")
df_diss = pd.read_csv(diss_path)

print(df_diss)

   utterance_id     aro_s     val_s     aro_t     val_t  delta_arousal  \
0             1  0.155329 -0.674792  0.248168 -0.242050       0.092839   
1             2 -0.080474  0.001581  0.286293 -0.248449       0.366767   
2             3  0.735761  0.534915  0.427576  0.519725       0.308185   
3             4 -0.241028  0.075200  0.131011  0.173824       0.372038   

   delta_valence  dissonant_arousal  dissonant_valence  dissonant_any  
0       0.432742              False              False          False  
1       0.250030              False              False          False  
2       0.015191              False              False          False  
3       0.098624              False              False          False  


## ต่อไปเอา text มาติดด้วย

In [4]:
# ถ้าคุณมี dialogue_2_vad_text.csv พร้อม text อยู่แล้ว:
text_vad_path = os.path.join(OWN_SCRIPT_DIR, "dialogue_2_vad_text.csv")
df_text = pd.read_csv(text_vad_path)

# เอาเฉพาะ utterance_id + text
df_text = df_text[["utterance_id", "text"]]

# merge เข้ากับ df_diss
df = pd.merge(df_diss, df_text, on="utterance_id", how="left")

df = df.sort_values("utterance_id").reset_index(drop=True)
df


,utterance_id,aro_s,val_s,aro_t,val_t,delta_arousal,delta_valence,dissonant_arousal,dissonant_valence,dissonant_any,text
0,1,0.155329,-0.674792,0.248168,-0.242050,0.092839,0.432742,False,False,False,Why are you bothering me? What's the problem?
1,2,-0.080474,0.001581,0.286293,-0.248449,0.366767,0.250030,False,False,False,"Ahh that thing again, can you just stay away f..."
2,3,0.735761,0.534915,0.427576,0.519725,0.308185,0.015191,False,False,False,I'm fine! I am very good and doing well at the...
3,4,-0.241028,0.075200,0.131011,0.173824,0.372038,0.098624,False,False,False,"Besides, you are the one who seems to be doing..."


### ตั้ง OpenAI client

In [5]:
import getpass
import os

print("Setting up OpenAI client...")
if "OPENAI_API_KEY" not in os.environ:
    secret_key = getpass.getpass("Enter your OpenAI API key: ")
    os.environ["OPENAI_API_KEY"] = secret_key

client = OpenAI()
OPENAI_MODEL_ID = "gpt-4o-mini"
print("✅ OpenAI client ready:", OPENAI_MODEL_ID)


Setting up OpenAI client...
✅ OpenAI client ready: gpt-4o-mini


## Prompt templates ทั้ง 3 แบบ

### System prompt ร่วม (Luna Therapist)

In [6]:
SYSTEM_PROMPT = """You are "Luna", an empathetic CBT-oriented AI therapist. 
You respond warmly, validate emotions, and use gentle CBT strategies 
(reflection, thought-challenging, and collaborative planning).

Your answers must:
- Be 2–4 sentences.
- Be concrete and supportive.
- End with one open-ended question.
"""


### Base prompting (text only)

In [7]:
BASE_USER_TEMPLATE = """The client just said:

"{text}"

As Luna, please respond directly to the client.
Do NOT mention that you are an AI or that you saw any extra data.
"""


### Emotion-Enhanced prompting (ใช้ VA แต่ไม่พูดถึง dissonance)

In [8]:
EMO_USER_TEMPLATE = """The client just said:

"{text}"

We also estimated the client's emotional state on a valence-arousal scale:
- Text-based estimate: valence={val_t:.3f}, arousal={aro_t:.3f}
- Voice-based estimate: valence={val_s:.3f}, arousal={aro_s:.3f}

Use this information implicitly to guide your tone and content
(e.g., acknowledge high arousal or low valence),
but do NOT explicitly mention numbers or the measurement process.
Respond as Luna directly to the client.
"""


### Dissonance-Based prompting (เน้น mismatch)

In [9]:
DIS_USER_TEMPLATE = """The client just said:

"{text}"

We estimated the client's emotional state on a valence-arousal scale:
- Text-based estimate: valence={val_t:.3f}, arousal={aro_t:.3f}
- Voice-based estimate: valence={val_s:.3f}, arousal={aro_s:.3f}

There is an emotional dissonance between what the client says in words
and how they sound in their voice:
- Difference in arousal = {delta_arousal:.3f}
- Difference in valence = {delta_valence:.3f}
Dissonance flag = {dissonant_any}

Assume this mismatch might mean the client is downplaying or masking 
some feelings, or that their voice conveys more intensity than their words.

In your reply as Luna:
- Gently acknowledge both the content and the possible hidden/emotional layer.
- Validate the feelings that might not be fully stated.
- If appropriate, check for misunderstanding or minimization.
- Then continue with a brief CBT-style exploration.

Do NOT mention the words "dissonance", "valence", "arousal", or any numbers.
Speak naturally as a therapist to the client.
"""


## Loop รันทั้ง 3 conditions สำหรับ dialogue_2

In [10]:
results = []

for _, row in df.iterrows():
    utt_id = int(row["utterance_id"])
    text = row["text"]

    aro_s = float(row["aro_s"])
    val_s = float(row["val_s"])
    aro_t = float(row["aro_t"])
    val_t = float(row["val_t"])
    delta_a = float(row["delta_arousal"])
    delta_v = float(row["delta_valence"])
    diss_any = bool(row["dissonant_any"])

    # 1) Base
    base_user = BASE_USER_TEMPLATE.format(text=text)
    base_msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": base_user},
    ]
    try:
        base_completion = client.chat.completions.create(
            model=OPENAI_MODEL_ID,
            messages=base_msgs,
            max_tokens=256,
            temperature=0.6,
            top_p=0.9,
        )
        base_reply = base_completion.choices[0].message.content.strip()
    except Exception as e:
        base_reply = f"ERROR: {e}"

    # 2) Emotion-Enhanced
    emo_user = EMO_USER_TEMPLATE.format(
        text=text,
        val_t=val_t,
        aro_t=aro_t,
        val_s=val_s,
        aro_s=aro_s,
    )
    emo_msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": emo_user},
    ]
    try:
        emo_completion = client.chat.completions.create(
            model=OPENAI_MODEL_ID,
            messages=emo_msgs,
            max_tokens=256,
            temperature=0.6,
            top_p=0.9,
        )
        emo_reply = emo_completion.choices[0].message.content.strip()
    except Exception as e:
        emo_reply = f"ERROR: {e}"

    # 3) Dissonance-Based
    dis_user = DIS_USER_TEMPLATE.format(
        text=text,
        val_t=val_t,
        aro_t=aro_t,
        val_s=val_s,
        aro_s=aro_s,
        delta_arousal=delta_a,
        delta_valence=delta_v,
        dissonant_any=diss_any,
    )
    dis_msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": dis_user},
    ]
    try:
        dis_completion = client.chat.completions.create(
            model=OPENAI_MODEL_ID,
            messages=dis_msgs,
            max_tokens=256,
            temperature=0.6,
            top_p=0.9,
        )
        dis_reply = dis_completion.choices[0].message.content.strip()
    except Exception as e:
        dis_reply = f"ERROR: {e}"

    results.append({
        "utterance_id": utt_id,
        "client_text": text,
        "aro_s": aro_s,
        "val_s": val_s,
        "aro_t": aro_t,
        "val_t": val_t,
        "delta_arousal": delta_a,
        "delta_valence": delta_v,
        "dissonant_any": diss_any,
        "reply_base": base_reply,
        "reply_emotion": emo_reply,
        "reply_dissonance": dis_reply,
    })

len(results)


4

## เซฟเป็น CSV / JSONL ไว้ใช้ต่อ

In [11]:
out_df = pd.DataFrame(results)
out_csv = os.path.join(OWN_SCRIPT_DIR, "dialogue_2_replies_all_conditions.csv")
out_df.to_csv(out_csv, index=False)
print("Saved:", out_csv)

# เผื่ออยากเก็บเป็น JSONL สำหรับ future training
out_jsonl = os.path.join(OWN_SCRIPT_DIR, "dialogue_2_replies_all_conditions.jsonl")
with open(out_jsonl, "w", encoding="utf-8") as f:
    for rec in results:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
print("Saved:", out_jsonl)


Saved: /home/patsakornt/work/test/dissonance/own_script/dialogue_2/dialogue_2_replies_all_conditions.csv
Saved: /home/patsakornt/work/test/dissonance/own_script/dialogue_2/dialogue_2_replies_all_conditions.jsonl
